In [1]:
#클러스터 가중치 자동계산 함수 (Python)
# log-scaling vs softmax-scaling 비교
# GPT prompting template에서 가중치를 어떻게 사용해야 distractor가 자연스럽게 나오는지 예시

In [2]:
%%writefile preprocess_responses.py
import re
import numpy as np
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sentence_transformers import SentenceTransformer
import hdbscan
from sklearn.metrics.pairwise import cosine_similarity


def clean_response(text):
    if text is None:
        return None
    text = text.strip()
    if text.lower() in ["idk", "i don't know", "no idea", "??", "", "none"]:
        return None
    text = re.sub(r"[^\w\s.,'’-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if len(text) > 2 else None


t5_model = T5ForConditionalGeneration.from_pretrained("t5-base")
t5_tok = T5Tokenizer.from_pretrained("t5-base")


def canonicalize(text):
    inp = "paraphrase: " + text
    tokens = t5_tok(inp, return_tensors="pt", truncation=True, max_length=64, padding=True)
    out = t5_model.generate(**tokens, max_length=48)
    return t5_tok.decode(out[0], skip_special_tokens=True)


embed_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

def cluster_responses(responses, min_cluster_size=10):
    embeddings = embed_model.encode(responses, normalize_embeddings=True)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean')
    labels = clusterer.fit_predict(embeddings)
    return embeddings, labels


def representative_indices(embeddings, labels):
    reps = {}
    for cluster_id in set(labels):
        if cluster_id == -1:
            continue
        idxs = np.where(labels == cluster_id)[0]
        cluster_emb = embeddings[idxs]
        centroid = cluster_emb.mean(axis=0)
        sims = cosine_similarity([centroid], cluster_emb)[0]
        rep = idxs[np.argmax(sims)]
        reps[cluster_id] = rep
    return reps


def compute_weights(labels):
    unique, counts = np.unique(labels[labels != -1], return_counts=True)
    raw_counts = dict(zip(unique, counts))
    log_counts = {cid: np.log(1 + c) for cid, c in raw_counts.items()}
    total = sum(log_counts.values())
    weights = {cid: log_counts[cid] / total for cid in log_counts}
    return weights


def preprocess(raw_responses):
    cleaned = [clean_response(r) for r in raw_responses]
    cleaned = [r for r in cleaned if r is not None]
    canonical = [canonicalize(r) for r in cleaned]
    embeddings, labels = cluster_responses(canonical)
    reps_idx = representative_indices(embeddings, labels)
    reps = {cid: canonical[idx] for cid, idx in reps_idx.items()}
    weights = compute_weights(labels)
    return {
        "canonical_responses": canonical,
        "labels": labels,
        "cluster_representatives": reps,
        "cluster_weights": weights
    }


Overwriting preprocess_responses.py


In [3]:
import pandas as pd
from preprocess_responses import preprocess   # 방금 만든 파일

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

C:\Users\Hyemi\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hyemi\.cache\huggingface\hub\models--t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
response

,Respondent Id,ABTest_P[Q1],ABTest_P[Q2],Sports_S[Q1],Sports_S[Q2],Sports_P[Q1],Sports_P[Q2],Sandwiches[Q1],Sandwiches[Q2],Sandwiches[Q3],Electricity[Q1],Electricity[Q2],Electricity[Q3]
0,15617.0,NaN,NaN,b,.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,15491.0,NaN,NaN,b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,15440.0,NaN,NaN,b,it is not statistically significant because th...,c,idk,a,The piThe proportion of the picturesture of Th...,because it shows it as so much bigger just in ...,NaN,NaN,NaN
3,15684.0,NaN,NaN,NaN,NaN,NaN,NaN,b,NaN,NaN,NaN,NaN,NaN
4,15627.0,NaN,NaN,a,NaN,NaN,NaN,b,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4205,NaN,NaN,NaN,a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4206,NaN,NaN,NaN,a,I chose this answer because it gives a differe...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4207,NaN,NaN,NaN,a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4208,NaN,NaN,NaN,c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
response=pd.read_excel("data/Spring2023_DDM.xlsx")

Sports_S_Q2 = response["Sports_S[Q2]"].astype(str).tolist()
result_Sports_S_Q2 = preprocess(Sports_S_Q2)

Sandwiches_Q2 = response["Sandwiches[Q2]"].astype(str).tolist()
result_Sandwiches_S_Q2 = preprocess(Sandwiches_Q2)

C:\Users\Hyemi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Hyemi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Hyemi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Hyemi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
print("클러스터 대표 응답:")
for cid, rep in result["cluster_representatives"].items():
    print(f"Cluster {cid}: {rep}")

print("\n클러스터 가중치:")
for cid, w in result["cluster_weights"].items():
    print(f"Cluster {cid}: {w:.3f}")

In [ ]:
representatives = result["cluster_representatives"]
weights = result["cluster_weights"]